## Introduction

In this tutorial, we will build a character-level text autocomplete model using a Recurrent Neural Network (RNN) in PyTorch. We will train the model on the text from "warandpeace.txt". This project will help you understand how RNNs can be implemented for text generation tasks and their application in building your own autocomplete model.


## Importing Necessary Libraries

In [132]:
# This is Cell #1

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import random
import re


## Setting Up the Device

In [133]:
# This is Cell #2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## Reading and Preprocessing the Data

Now it is time to prepare our training data.


In [134]:
# This is Cell #3

def read_file(filename):
    with open(filename, "r", encoding="utf-8") as file:
        text = file.read().lower()
        # Keep only lowercase letters and standard punctuation (.,!?;:()[])
        text = re.sub(r'[^a-z.,!?;:()\[\] ]+', '', text)
    return text

sequence = read_file("/Users/<redacted>/Desktop/cs383/proj/assignment-7-rnns-ml-complete-<redacted>/warandpeace.txt")


### Here we will train our model with a simple sequence

We will start by training our model with a simple sequence and repettitive sequence such as `"abcdefghijklmnopqrstuvwxyzabcdef..."`, and we will see if our RNN is capable of learning that pattern or not. This will help you easily verify if your RNN is working correctly or not.

In [135]:
# This is Cell #4

#sequence = "abcdefghijklmnopqrstuvwxyz" * 100

## Create Character Mappings

Creating character mappings is essential because RNNs require numerical input to process data. By mapping each unique character to an index and creating a reverse mapping, we convert text data into numerical sequences that the model can understand. This step allows us to encode input text for training and decode the model's output back into readable characters during text generation.



In [136]:
# This is Cell #5

#TODO: Create a list of unique characters from the text sequence
vocab = sorted(set(sequence))

#TODO: Create two dictionaries for character-index mappings that map each character in vocab to a unique index and vice versa
char_to_idx = {char: idx for idx, char in enumerate(vocab)}
idx_to_char = {idx: char for idx, char in enumerate(vocab)}

#TODO: Convert the entire text based data into numerical data
data = [char_to_idx[char] for char in sequence]


## Defining the CharDataset Class

Now we will create a custom dataset class to generate sequences and targets for training

Creating a custom `CharDataset` class is crucial because it prepares our text data into input sequences and target sequences that the RNN can learn from. By organizing the data this way, we can efficiently feed batches of sequences into the model during training, allowing it to learn the patterns of character sequences in the text.

In [137]:
# This is Cell #6

class CharDataset(Dataset):
    def __init__(self, data, sequence_length, stride, vocab_size):
        self.data = data
        self.sequence_length = sequence_length
        self.stride = stride
        self.vocab_size = vocab_size
        self.sequences = []
        self.targets = []
        
        # Create overlapping sequences with stride
        for i in range(0, len(data) - sequence_length, stride):
            self.sequences.append(data[i:i + sequence_length])
            self.targets.append(data[i + 1:i + sequence_length + 1])

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
        target = torch.tensor(self.targets[idx], dtype=torch.long)
        return sequence, target
    

## Setting Hyperparameters

Now we will set our model's hyperparameters for our training process

Setting hyperparameters is important because they define the model's architecture and training behavior. They determine how the RNN processes data, learns patterns, and how quickly it converges during training. Properly chosen hyperparameters can significantly improve model performance and is a key step in training of models

Set the following hyperparameters for your model in the code cell below:
`sequence_length`, `stride`, `embedding_dim`, `hidden_size`, `num_layers`, `learning_rate`, `num_epochs`, `batch_size`, `vocab_size`.

In [ ]:
# This is Cell #7

#TODO: Set your model's hyperparameters

sequence_length = 100
stride = 10
embedding_dim = 64
hidden_size = 256
learning_rate = 0.0001
num_epochs = 20
batch_size = 64
vocab_size = len(vocab)     
input_size = len(vocab)     
output_size = len(vocab)   

'\nsequence_length = 1000  # Length of each input sequence\nstride = 10            # Stride for creating sequences\nembedding_dim = 2     # Dimension of character embeddings\nhidden_size = 1      # Number of features in the hidden state of the RNN\nlearning_rate = 200  # Learning rate for the optimizer\nnum_epochs = 1         # Number of epochs to train\nbatch_size = 64        # Batch size for training\nvocab_size = len(vocab)\ninput_size = len(vocab)\noutput_size = len(vocab)\n\n'

After you have set your hyperparameters in the code cell above, very breifly tell what is the role of each of the hyperparameter that you have defined above.

TODO: Explain below
Here's a brief explanation of the role of each hyperparameter:

1. **sequence_length = 100**: Defines the length of input sequences. The model will process sequences of 100 tokens (words or characters) at a time.

2. **stride = 10**: The step size for moving through the dataset during training. A stride of 10 means the model will move 10 tokens forward after processing each sequence.

3. **embedding_dim = 64**: The dimensionality of the embedding vectors. Each token from the vocabulary will be represented as a 64-dimensional vector.

4. **hidden_size = 256**: The size of the hidden state in the recurrent layers (e.g., LSTM or GRU). It determines the model's capacity to learn and represent patterns in the data.

5. **learning_rate = 0.0001**: The step size used in gradient descent to update the model’s parameters. A smaller learning rate ensures more gradual learning.

6. **num_epochs = 20**: The number of times the entire training dataset will be passed through the model. More epochs allow the model to learn better but may also lead to overfitting.

7. **batch_size = 64**: The number of samples processed together in one forward/backward pass. A larger batch size can speed up training but may use more memory.

8. **vocab_size = len(vocab)**: The number of unique tokens (words or characters) in the vocabulary. It determines the input and output dimensions of the model.

9. **input_size = len(vocab)**: The size of the input layer, which is the same as the vocabulary size, meaning the model expects inputs that are represented as one-hot vectors of this size.

10. **output_size = len(vocab)**: The size of the output layer, which is also equal to the vocabulary size, as the model predicts a token from the vocabulary at each timestep.

## Splitting Data into Training and Testing Sets

By now at this point in class, I'm confident that you know why we do this, so I'm not gonna say a lot here, let's jump right into the todo.

In [139]:
# This is Cell #8

data_tensor = torch.tensor(data, dtype=torch.long)

#TODO: Convert the data into a pytorch tensor and split the data into 90:10 ratio
train_size = int(0.9 * len(data_tensor))
train_data = data_tensor[:train_size]
test_data = data_tensor[train_size:]


## Creating Data Loaders

Now we will create data loaders for easy batching during training and testing.

Creating data loaders is essential to batch the data during training and testing. Batching allows the RNN to process multiple sequences in parallel, which speeds up training and makes better use of computational resources. 
We will also use Data loaders to shuffle the batched data, which is important for training models that generalize well.

Make sure to set `drop_last=True`

In [140]:
# This is Cell #9

train_dataset = CharDataset(train_data, sequence_length, stride, vocab_size)
test_dataset = CharDataset(test_data, sequence_length, stride, vocab_size)

#TODO: Initialize the training and testing data loader with batching and shuffling equal to True for training (and shuffling = False for testing)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

total_batches = len(train_loader)



## Defining the RNN Model

Here we will define our character-level RNN model.

In [141]:
# This is Cell #10

class CharRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, embedding_dim=30):
        super(CharRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = torch.nn.Embedding(output_size, embedding_dim)
        self.W_e = nn.Parameter(torch.randn(hidden_size, embedding_dim) * 0.01)  # Smaller std
        self.b_e = nn.Parameter(torch.zeros(hidden_size))
        self.W_h = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)  # Smaller std
        self.b_h = nn.Parameter(torch.zeros(hidden_size)) 
        #TODO: set the fully connected layer
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden):
        """
        x in [b, l] # b is batch_size and l is sequence length
        """
        x_embed = self.embedding(x)  # [b=batch_size, l=sequence_length, e=embedding_dim]
        b, l, _ = x_embed.size()
        x_embed = x_embed.transpose(0, 1) # [l, b, e]
        if hidden is None:
            h_t_minus_1 = self.init_hidden(b)
        else:
            h_t_minus_1 = hidden
        output = []
        for t in range(l):
            # RNN equation from the lecture 
            # We add a bias as well to expand the range of learnable functions
            h_t = torch.tanh(x_embed[t] @ self.W_e.T + self.b_e + h_t_minus_1 @ self.W_h.T + self.b_h) # [b, e]
            output.append(h_t)
            h_t_minus_1 = h_t
        output = torch.stack(output) # [l, b, e]
        output = output.transpose(0, 1) # [b, l, e]
        final_hidden = h_t.clone() # [b, h]
        logits = self.fc(output) # [b, l, vocab_size=v] 
        return logits, final_hidden
    
    def init_hidden(self, batch_size):
        return torch.zeros(batch_size, self.hidden_size).to(device)


For a basic high level understanding of what is the CharRNN model that you just defined above, it consists of an embedding layer, an RNN layer, and a fully connected layer. Then embedding layer converts character indices into embeddings. Then RNN processes the embeddings and captures sequential information. Then finally the fully connected layer maps the RNN outputs to the vocabulary size for character prediction.


# Initializing the Model, Loss Function, and Optimizer

Now we will create an instance of the model that we just defined above and set up the loss function and optimizer. Then we will define a loss function, that evaluates the model's prediction against the true targets, and attaches a cost (number) on how good/bad the model is doing. During our training process, it is this cost that we try to minimize by tweaking the weights of the network. 

Then we will set up an optimizer, which will update the model's parameters based on the loss returned by the our loss function. This is how our model will learn over time.


In [142]:
# This is Cell #12

#TODO: Initialize your RNN model
model = CharRNN(input_size, hidden_size, output_size, embedding_dim).to(device)

#TODO: Define the loss function (use cross entropy loss)
criterion = nn.CrossEntropyLoss()

#TODO: Initialize your optimizer passing your model parameters and training hyperparameters
optimizer = optim.Adam(model.parameters(), lr=learning_rate)


## Training the Model

Now finally, after all the setup that we have done, we can train our RNN. 

A basic idea high level idea of what we will do here is we will loop over epochs and batches to train the model. 
We will Initialize the hidden state at the beginning of each epoch. For each batch, we will reset the gradients, perform a forward pass, compute the loss, perform backpropagation, and update the model parameters. Then we detach the hidden state to prevent gradients from backpropagating through previous batches. We ill repeat this process for each batch. And finally we will calculate the average loss and accuracy for each epoch.
By performing forward and backward passes, calculating loss, and updating the model parameters, we enable the RNN to improve its predictions with each epoch.

In [149]:
# This is Cell #13

for epoch in range(num_epochs):
    total_loss, correct_predictions, total_predictions = 0, 0, 0

    hidden = model.init_hidden(batch_size)

    for batch_idx, (batch_inputs, batch_targets) in tqdm(enumerate(train_loader), total=total_batches, desc=f"Epoch {epoch+1}/{num_epochs}"):
        batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)

        output, hidden = model(batch_inputs, hidden)

        hidden = hidden.detach()

        loss = criterion(output.view(-1, output_size), batch_targets.view(-1))  # Flatten the outputs and targets for CrossEntropyLoss
        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        with torch.no_grad():
            # Calculate accuracy
            _, predicted_indices = torch.max(output, dim=2)  # Predicted characters

            correct_predictions += (predicted_indices == batch_targets).sum().item()
            total_predictions += batch_targets.size(0) * batch_targets.size(1)  # Total items in this batch

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    accuracy = correct_predictions / total_predictions * 100  # Convert to percentage
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")

Epoch 1/20:   0%|          | 0/4387 [00:00<?, ?it/s]/var/folders/9x/hw3zm4hn0db63kdhrpwp6j2h0000gn/T/ipykernel_1841/1273917643.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
/var/folders/9x/hw3zm4hn0db63kdhrpwp6j2h0000gn/T/ipykernel_1841/1273917643.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target = torch.tensor(self.targets[idx], dtype=torch.long)
Epoch 1/20: 100%|██████████| 4387/4387 [02:07<00:00, 34.52it/s]


Epoch [1/20], Loss: 1.3617, Accuracy: 59.69%


Epoch 2/20: 100%|██████████| 4387/4387 [02:15<00:00, 32.49it/s]


Epoch [2/20], Loss: 1.3585, Accuracy: 59.78%


Epoch 3/20: 100%|██████████| 4387/4387 [02:29<00:00, 29.29it/s]


...

Epoch [18/20], Loss: 1.3453, Accuracy: 60.13%


Epoch 19/20: 100%|██████████| 4387/4387 [02:16<00:00, 32.07it/s]


Epoch [19/20], Loss: 1.3452, Accuracy: 60.14%


Epoch 20/20: 100%|██████████| 4387/4387 [02:04<00:00, 35.14it/s]

Epoch [20/20], Loss: 1.3448, Accuracy: 60.15%


## Check your loss

The training loss of your model when trained with a simple sequence like `"abcdefghijklmnopqrstuvwxyz" * 100` should be extremely close to zero. If that's not the case, go back and fix your bugs ;)

If you have acheived a training loss of 0 or extremley close to 0, then congratulations, lets move on to train your model with a bit more complicated sequence. That is our old favorite book, `warandpeace.txt`.

### Read the `warandpeace.txt` file

In [144]:
# This is Cell #14

sequence = read_file('warandpeace.txt')

### Now Follow the instructions

1. Re-run Cell #5 to re-create character mappings for `warandpeace.txt`
2. Re-run Cell #7 to re-initialize hyperparameters
3. Re-run Cell #8 to split and create training and testing data with `warandpeace.txt` as your corpus
4. Re-run Cell #9 to set up data loaders with `warandpeace.txt` data
5. Re-run Cell #12 to re-initialize a new model object (maybe ask yourself why can't you use the previous model that was trained on the simple `"abc..."` corpus)
6. Re-run Cell #13 to train the new model with `warandpeace.txt` data.
   

## Evaluating the Model

After training, we evaluate the model on the test data.

In [150]:
# This is Cell #15

with torch.no_grad():
    #TODO: Write the testing loop for your trained model by refering to the training loop code given to you above
    model.eval()  # Set the model to evaluation mode
    total_loss, correct_predictions, total_predictions = 0, 0, 0

    for batch_idx, (batch_inputs, batch_targets) in tqdm(enumerate(test_loader), total=total_batches, desc="Testing"):
        batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)

        # Forward pass
        output, _ = model(batch_inputs, None)  # No need to provide hidden state for testing

        # Calculate loss
        loss = criterion(output.view(-1, output_size), batch_targets.view(-1))  # Flatten the outputs and targets for CrossEntropyLoss

        # Calculate accuracy
        _, predicted_indices = torch.max(output, dim=2)  # Predicted characters

        correct_predictions += (predicted_indices == batch_targets).sum().item()
        total_predictions += batch_targets.size(0) * batch_targets.size(1)  # Total items in this batch

        total_loss += loss.item()

    avg_loss = total_loss / len(test_loader)
    accuracy = correct_predictions / total_predictions * 100  # Convert to percentage


    print(f"Test Loss: {avg_loss:.4f}, Test Accuracy: {accuracy:.2f}%")

Testing:   0%|          | 0/4387 [00:00<?, ?it/s]/var/folders/9x/hw3zm4hn0db63kdhrpwp6j2h0000gn/T/ipykernel_1841/1273917643.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
/var/folders/9x/hw3zm4hn0db63kdhrpwp6j2h0000gn/T/ipykernel_1841/1273917643.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target = torch.tensor(self.targets[idx], dtype=torch.long)
Testing:  11%|█         | 487/4387 [00:04<00:33, 117.64it/s]

Test Loss: 1.4519, Test Accuracy: 57.28%


## Generating Text with the Trained Model

In this part of the assignment, your task is to implement the `generate_text` function, which uses a trained RNN model to generate text character-by-character, continuing from a given input. The function will produce an extended sequence by repeatedly predicting and appending the next character to the input.

### What the function is supposed to do?

1. Take an initial input text of length `n` from the user, convert it into indices using a predefined vocabulary (char_to_idx).
2. Use a trained model to predict the next character in the sequence.
3. Append the predicted character to the input, extend the input sequence, and repeat the process until `k` additional characters are generated.
4. Return the generated text, including the original input and the newly predicted characters.


In [157]:
# This is Cell #16

def sample_from_output(logits, temperature=1.0):
    """
    Sample from the logits with temperature scaling.
    logits: Tensor of shape [batch_size, vocab_size] (raw scores, before softmax)
    temperature: a float controlling the randomness (higher = more random)
    """
    # Apply temperature scaling to logits (increase randomness with higher values)
    scaled_logits = logits / temperature  # Scale the logits by temperature
    # Apply softmax to convert logits to probabilities
    probabilities = F.softmax(scaled_logits, dim=1)
    
    # Sample from the probability distribution
    sampled_idx = torch.multinomial(probabilities, 1)  # Sample one index from the probability distribution
    return sampled_idx

def generate_text(model, start_text, n, k, temperature=1.0):
    """
        model: The trained RNN model used for character prediction.
        start_text: The initial string of length `n` provided by the user to start the generation.
        n: The length of the initial input sequence.
        k: The number of additional characters to generate.
        temperature: Optional
        A scaling factor for randomness in predictions. Higher values (e.g., >1) make 
            predictions more random, while lower values (e.g., <1) make predictions more deterministic.
            Default is 1.0.
    """
    start_text = start_text.lower()
    #TODO: Implement the rest of the generate_text function
    model.eval()  # Set the model to evaluation mode
    device = next(model.parameters()).device  # Ensure the model's device is used

    # Convert start_text to indices using char_to_idx
    input_indices = [char_to_idx[char] for char in start_text]
    input_tensor = torch.tensor([input_indices], dtype=torch.long, device=device)

    # Initialize hidden state (without device argument)
    hidden = model.init_hidden(1)

    generated_text = start_text

    with torch.no_grad():  # Disable gradient calculation
        for _ in range(k):
            output, hidden = model(input_tensor, hidden)
            logits = output[0, -1].unsqueeze(0)
            sampled_idx = sample_from_output(logits, temperature)
            next_char = idx_to_char[sampled_idx.item()]

            # Append the next character to the generated text
            generated_text += next_char

            # Update input_tensor with the last predicted character
            input_tensor = torch.tensor([[sampled_idx.item()]], dtype=torch.long, device=device)

    return generated_text
    #return generated_text

print("Training complete. Now you can generate text.")
while True:
    start_text = input("Enter the initial text (n characters, or 'exit' to quit): ")
    
    if start_text.lower() == 'exit':
        print("Exiting...")
        break
    
    n = len(start_text) 
    k = int(input("Enter the number of characters to generate: "))
    temperature_input = input("Enter the temperature value (1.0 is default, >1 is more random): ")
    temperature = float(temperature_input) if temperature_input else 1.0
    
    completed_text = generate_text(model, start_text, n, k, temperature)
    
    print(f"Generated text: {completed_text}")

Training complete. Now you can generate text.
Generated text: the mason looked intently at pierre fall sounded, ask t
Generated text: on reaching petersburg pierre did not go there into her son.ah
Generated text: the mason looked intently at pierre. the countess and t
Generated text: the mason looked intently at pierre, and not fursh and 
Generated text: the mason looked intently at pierreceldstagiflaws.ihapp


ValueError: invalid literal for int() with base 10: ''

## Report section

In your report, describe your experiments and observations when training the model with two datasets: (1) the sequence `"abcdefghijklmnopqrstuvwxyz" * 100` and (2) the text from `warandpeace.txt`. Include the final loss values for both datasets and discuss how the generated text differed between the two. Explain the impact of changing the `temperature` parameter on the text generation, and provide examples. Reflect on the challenges you faced, your thought process during implementation, and the key insights you gained about RNNs and sequence modeling.

The sequence output was: Test Loss: 0.014, Test Accuracy: 99.96%.
The sequence was simple and easy to understand for the RNN and gave good results.
warandpeace.txt output: Test Loss: 1.3448, Test Accuracy: 60.15%.
The complexity of the text made it harder for the model to predict the next character accurately, highlighting the challenges of training on large, diverse d atasets.
The model performed better for the sequence than the larger warandpeace text. The generated text for the sequence was usually the next character or the one after that and so on and so forth, whereas the generated text from warandpeace.txt made much more sense, however, had mistakes.
Impact of temperature: The temperature parameter influenced text generation significantly. At temperature 1.0, the model generated varied text with some randomness.
Higher temperatures (e.g., 1.5) made the text more unpredictable, whilelower temperatures (e.g., 0.5) caused the model to repeat common patterns, producing more coherent but less creative sequences. 
Ex. Using prompt "the mason looked intently at pierre", a temperature of 0.2 gave us the response "the mason looked intently at pierre. the countess and t" which is much more in line with the text than the response with temperature 1 which was "the mason looked intently at pierre, and not fursh and ". A temperature of 2 gave us a very nonsensical output of "the mason looked intently at pierreceldstagiflaws.ihapp" showing how unpredictable the response gets as we increase tmperature.
I faced challenges with the hyperparameter tuning and achieved a maximum accuracy of 60.15 percent after 5 hours of tuning. I triedadjust the hyperparamters to see how the eccuracy would respond and use my inference to make adjustments. I gained a key insight into RNNs about how hyperparamaters can either make or break an entire model.


